## 2D Convolution

In [1]:
%%writefile q11.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define IMG_WIDTH  8
#define IMG_HEIGHT 8
#define LOW 10
#define HIGH 99
#define BLOCK_SIZE 8
#define MASK_WIDTH 3

// ── CUDA Kernel
__global__ void conv2DKernel(int *dN, int *dM, int *dP,
                              int imgWidth, int imgHeight, int maskWidth)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (col < imgWidth && row < imgHeight) {
        int dPvalue = 0;
        for (int r = 0; r < maskWidth; r++) {
            for (int c = 0; c < maskWidth; c++) {
                int N_row = row + r;
                int N_col = col + c;
                if (N_row < imgHeight && N_col < imgWidth)
                    // Flipped mask indices (180 degree rotation)
                    dPvalue += dN[N_row * imgWidth + N_col]
                             * dM[(maskWidth-1-r) * maskWidth + (maskWidth-1-c)];
            }
        }
        dP[row * imgWidth + col] = dPvalue;
    }
}

// ── Host Function
__host__ void conv2D(int *hN, int *hM, int *hP,
                     int imgWidth, int imgHeight, int maskWidth)
{
    int *dN, *dM, *dP;
    int sizeN = imgWidth * imgHeight * sizeof(int);
    int sizeM = maskWidth * maskWidth * sizeof(int);

    cudaMalloc((void**)&dN, sizeN);
    cudaMalloc((void**)&dM, sizeM);
    cudaMalloc((void**)&dP, sizeN);

    cudaMemcpy(dN, hN, sizeN, cudaMemcpyHostToDevice);
    cudaMemcpy(dM, hM, sizeM, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)imgWidth  / BLOCK_SIZE),
                 (int)ceil((float)imgHeight / BLOCK_SIZE), 1);

    conv2DKernel<<<DimGrid, DimBlock>>>(dN, dM, dP, imgWidth, imgHeight, maskWidth);

    cudaMemcpy(hP, dP, sizeN, cudaMemcpyDeviceToHost);

    cudaFree(dN); cudaFree(dM); cudaFree(dP);
}

// ── Main
int main()
{
    int hN[IMG_HEIGHT][IMG_WIDTH];
    int hP[IMG_HEIGHT][IMG_WIDTH];
    int hM[MASK_WIDTH][MASK_WIDTH] = {{1,2,3},{4,5,6},{7,8,9}};

    srand(time(NULL));
    for (int r = 0; r < IMG_HEIGHT; r++)
        for (int c = 0; c < IMG_WIDTH; c++)
            hN[r][c] = rand() % (HIGH - LOW + 1) + LOW;

    printf("hN (Image %dx%d):\n", IMG_HEIGHT, IMG_WIDTH);
    for (int r = 0; r < IMG_HEIGHT; r++) {
        for (int c = 0; c < IMG_WIDTH; c++) printf("%4d", hN[r][c]);
        printf("\n");
    }

    printf("\nhM (Mask %dx%d):\n", MASK_WIDTH, MASK_WIDTH);
    for (int r = 0; r < MASK_WIDTH; r++) {
        for (int c = 0; c < MASK_WIDTH; c++) printf("%4d", hM[r][c]);
        printf("\n");
    }

    conv2D((int*)hN, (int*)hM, (int*)hP, IMG_WIDTH, IMG_HEIGHT, MASK_WIDTH);

    printf("\nhP (2D Convolution Result):\n");
    for (int r = 0; r < IMG_HEIGHT; r++) {
        for (int c = 0; c < IMG_WIDTH; c++) printf("%6d", hP[r][c]);
        printf("\n");
    }

    return 0;
}

Writing q11.cu


In [2]:
!nvcc -arch=sm_75 q11.cu -o q11

!./q11

hN (Image 8x8):
  31  98  26  14  87  83  43  22
  32  22  32  55  71  20  91  17
  87  59  47  13  44  52  53  92
  62  98  34  78  54  44  89  75
  94  67  79  81  51  74  55  35
  87  40  80  20  50  71  79  90
  30  27  93  64  69  46  19  31
  44  95  61  88  91  13  64  85

hM (Mask 3x3):
   1   2   3
   4   5   6
   7   8   9

hP (2D Convolution Result):
  2101  1984  1917  2347  2927  2242  1537   576
  2109  1920  2060  2095  2511  2357  2150   930
  3077  2508  2095  2166  2439  2912  2357  1383
  3216  3121  2721  2710  2728  2953  2323  1155
  3214  2853  2972  2685  2725  2739  1818   948
  2733  2518  2834  2449  2654  2648  2062  1251
  2120  2662  3002  2485  1989  1521  1228   789
  1583  1959  1890  1611  1371  1224  1256   765
